In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD

# 1. Muat Data
df_train = pd.read_csv('../data/processed/train.csv')

# 2. Siapkan Model Popularity (Sebagai Fallback / Cadangan)
# Hitung item yang paling sering berinteraksi
popular_items = df_train['item_id'].value_counts().index.tolist()

# 3. Siapkan Model Collaborative Filtering (SVD)
user_item_matrix = df_train.pivot(index='user_id', columns='item_id', values='rating').fillna(0)
svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_
df_cf_preds = pd.DataFrame(np.dot(user_factors, item_factors), index=user_item_matrix.index, columns=user_item_matrix.columns)

print("Komponen Candidate Generation siap digunakan!")

Komponen Candidate Generation siap digunakan!


In [2]:
def generate_candidates(user_id, num_candidates=100):
    """
    Fungsi untuk menghasilkan pool kandidat gabungan.
    Strategi: 50% dari CF, 50% dari Popularity.
    """
    candidate_pool = set()
    
    # 1. Tarik Kandidat dari Collaborative Filtering (Jika user bukan Cold-Start)
    if user_id in df_cf_preds.index:
        cf_quota = int(num_candidates * 0.5) # Ambil jatah 50 item
        cf_top = df_cf_preds.loc[user_id].sort_values(ascending=False).head(cf_quota).index.tolist()
        candidate_pool.update(cf_top)
        
    # 2. Penuhi sisa kuota dari Popularity Baseline
    # Terus tambahkan item terlaris sampai jumlah kandidat mencapai num_candidates
    for item in popular_items:
        if len(candidate_pool) >= num_candidates:
            break
        candidate_pool.add(item)
        
    # 3. Hapus item yang sudah pernah berinteraksi dengan user
    user_history = set(df_train[df_train['user_id'] == user_id]['item_id'].tolist())
    final_candidates = list(candidate_pool - user_history)
    
    # Jika setelah dihapus jumlahnya kurang dari target, tambahkan lagi dari Popularity
    if len(final_candidates) < num_candidates:
        for item in popular_items:
            if item not in user_history and item not in final_candidates:
                final_candidates.append(item)
            if len(final_candidates) == num_candidates:
                break
                
    return final_candidates[:num_candidates]

# Mari kita uji pipa Candidate Generation ini!
user_test = 12
candidates = generate_candidates(user_id=user_test, num_candidates=100)

print(f"Total kandidat terkumpul untuk User {user_test}: {len(candidates)} item")
print(f"10 Sampel Kandidat Pertama: {candidates[:10]}")

Total kandidat terkumpul untuk User 12: 100 item
10 Sampel Kandidat Pertama: [1, 257, 258, 7, 8, 9, 135, 11, 12, 651]
